- [ ] Is pit stop duration genuinely different by team (a crew-skill effect), or does it wash out once accounting for stop count/race chaos? (ANOVA across teams, or team as a regression dummy variable)
- [ ] Does a slow pit stop reliably cost track position, or does pack density/traffic matter more? (correlate stop_duration against the position-swing data)
- [ ] Are disaster stops (Tukey-fence outliers) random, or concentrated in specific teams or circuits? (chi-square)

In [3]:
import sys
sys.path.append("../")
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm


import sqlite3

conn = sqlite3.connect("../DATA INGESTION/f1.db")


- [X] Is pit stop duration genuinely different by team (a crew-skill effect), or does it wash out once accounting for stop count/race chaos? (ANOVA across teams, or team as a regression dummy variable)

In [9]:
from data_prep import normalize_team_names


pit_query = """
SELECT p.session_key, p.driver_number, p.lap_number,
       p.stop_duration,
       gs.year, m.circuit_short_name,
       d.team_name
FROM silver_pit p
JOIN silver_sessions gs ON p.session_key = gs.session_key AND gs.session_name = 'Race'
JOIN silver_meetings m ON gs.meeting_key = m.meeting_key
JOIN silver_drivers d ON p.session_key = d.session_key AND p.driver_number = d.driver_number
WHERE p.stop_duration IS NOT NULL
"""
df_pits = pd.read_sql(pit_query, conn)
df_pits = normalize_team_names(df_pits)

df_pits.shape
df_pits['year'].value_counts()
df_pits['stop_duration'].describe()

count    922.000000
mean       3.680586
std        3.352602
min        1.800000
25%        2.400000
50%        2.700000
75%        3.300000
max       38.300000
Name: stop_duration, dtype: float64

In [10]:
q1 = df_pits['stop_duration'].quantile(0.25)
q3 = df_pits['stop_duration'].quantile(0.75)
iqr = q3 - q1

lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print(f"Q1: {q1:.3f}, Q3: {q3:.3f}, IQR: {iqr:.3f}")
print(f"Lower fence: {lower_fence:.3f}, Upper fence: {upper_fence:.3f}")
print(f"Rows above upper fence: {(df_pits['stop_duration'] > upper_fence).sum()}")

Q1: 2.400, Q3: 3.300, IQR: 0.900
Lower fence: 1.050, Upper fence: 4.650
Rows above upper fence: 106


In [11]:
df_pits_clean = df_pits[df_pits['stop_duration'] <= upper_fence].copy()
print(df_pits_clean.shape)
print(df_pits_clean['team_name'].value_counts())
print(df_pits_clean['stop_duration'].describe())

(816, 7)
team_name
Ferrari            90
Mercedes           90
McLaren            83
Haas F1 Team       80
RB Family          80
Red Bull Racing    80
Alpine             80
Williams           79
Aston Martin       78
Sauber Family      68
Cadillac            8
Name: count, dtype: int64
count    816.000000
mean       2.784314
std        0.555706
min        1.800000
25%        2.400000
50%        2.600000
75%        3.100000
max        4.600000
Name: stop_duration, dtype: float64


In [12]:
df_pits_clean = df_pits_clean[df_pits_clean['team_name'] != 'Cadillac'].copy()
df_pits_clean.shape

(808, 7)

In [13]:
from scipy import stats

groups = [g['stop_duration'].values for name, g in df_pits_clean.groupby('team_name')]
f_stat, p_value = stats.f_oneway(*groups)
print(f"Naive ANOVA: F={f_stat:.3f}, p={p_value:.6f}")

df_pits_clean.groupby('team_name')['stop_duration'].agg(['mean','median','std','count']).sort_values('mean')

Naive ANOVA: F=11.021, p=0.000000


,mean,median,std,count
team_name,,,,
Ferrari,2.461111,2.30,0.419924,90
RB Family,2.623750,2.50,0.472870,80
Red Bull Racing,2.626250,2.45,0.515272,80
Mercedes,2.687778,2.50,0.485249,90
McLaren,2.719277,2.40,0.699208,83
Sauber Family,2.807353,2.60,0.574928,68
Aston Martin,2.934615,2.70,0.545781,78
Williams,2.948101,2.90,0.499194,79
Alpine,2.980000,2.85,0.503758,80


In [16]:
sc_query = """
SELECT DISTINCT session_key, lap_number
FROM silver_race_control
WHERE category = 'SafetyCar' AND lap_number IS NOT NULL
"""
df_sc = pd.read_sql(sc_query, conn)
df_sc['sc_active'] = 1

df_pits_clean = df_pits_clean.merge(
    df_sc, on=['session_key','lap_number'], how='left'
)
df_pits_clean['sc_active'] = df_pits_clean['sc_active'].fillna(0).astype(int)
df_pits_clean['sc_active'].value_counts()

sc_active
0    696
1    112
Name: count, dtype: int64

In [17]:
import statsmodels.formula.api as smf

model_ancova = smf.ols(
    'stop_duration ~ C(team_name) + C(sc_active) + C(circuit_short_name)',
    data=df_pits_clean
).fit()
print(model_ancova.summary())

                            OLS Regression Results                            
Dep. Variable:          stop_duration   R-squared:                       0.162
Model:                            OLS   Adj. R-squared:                  0.126
Method:                 Least Squares   F-statistic:                     4.530
Date:                Sun, 19 Jul 2026   Prob (F-statistic):           5.00e-15
Time:                        11:09:31   Log-Likelihood:                -594.58
No. Observations:                 808   AIC:                             1257.
Df Residuals:                     774   BIC:                             1417.
Df Model:                          33                                         
Covariance Type:            nonrobust                                         
                                                  coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

METHOD:
Used stop_duration (the stationary time in the pit box -- the crew's actual task) 
as the outcome variable. Scoped to 2024+ due to zero coverage in 2023 (Notes item 
#7). Excluded disaster stops via a freshly-derived Tukey fence on the 2024+ 
sample: cutoff at 4.65s (very close to the earlier 4.9s from item #9 but re-
derived on this specific sample; 106 rows / ~11.5% flagged as disasters and 
excluded). Excluded Cadillac due to small n=8 (consistent with earlier questions).
Final n = 808 pit stops across 10 established teams.

Two-stage analysis matching the "washes out?" question structure:
  Stage 1: Naive one-way ANOVA on stop_duration by team.
  Stage 2: ANCOVA -- OLS regression with team_name, sc_active (Safety Car during 
           the stop's lap), and circuit_short_name as predictors. Tests whether 
           team effects survive controlling for race chaos and pit-lane 
           characteristics.

STAGE 1 RESULT: F=11.02, p<0.001 -- clearly significant. Team means range from 
Ferrari (2.46s) to Haas (3.05s), a spread of ~0.6 seconds.

STAGE 2 RESULT: R² = 0.162. Team effects survive controls:
  Ferrari:         -0.520s vs Alpine baseline, p<0.001 (clearly fastest)
  RB Family:       -0.350s, p<0.001
  Red Bull Racing: -0.350s, p<0.001 (matches RB Family -- shared technical 
                   infrastructure between the two teams)
  Mercedes:        -0.283s, p<0.001
  McLaren:         -0.251s, p=0.002
  Sauber Family:   -0.165s, p=0.055 (borderline)
  Aston Martin:    -0.025s, p=0.766 (indistinguishable from Alpine)
  Williams:        -0.028s, p=0.734 (indistinguishable from Alpine)
  Haas F1 Team:    +0.089s, p=0.279 (slower than Alpine but not significantly)

Two clear tiers emerge:
  - FAST TIER (Ferrari, RB, Red Bull, Mercedes, McLaren): significantly faster 
    than the baseline group, all p<0.005.
  - MIDDLE/SLOW TIER (Alpine, Aston Martin, Williams, Haas): statistically 
    indistinguishable from each other.
  - Sauber Family sits between.

Confounder behavior:
  - sc_active = +0.179s, p=0.004 -- pit stops under Safety Car take ~0.18s 
    LONGER, not shorter. Likely because SC stops often involve waiting for the 
    pack or double-stacked stops requiring queuing. Real, statistically 
    detectable confounder worth including.
  - Circuit dummies: mostly non-significant. Turns out pit-lane physical 
    characteristics don't meaningfully affect stationary time (which measures 
    only the crew's task, not lane drive time). Kept in the model as a control 
    but had minimal effect on team coefficients.

INTERPRETATION -- ANSWER TO THE "WASHES OUT" QUESTION:
The team crew-skill effect is REAL and does NOT wash out when controlling for 
race chaos (SC) and circuit. Ferrari's ~0.52s lead over Alpine (and larger lead 
over Haas/Williams/Aston Martin) is a genuine, isolable crew-skill effect, not 
an artifact of some teams pitting under different conditions than others.

Three independent findings emerged from the same model:
  1. Team crew skill is a real, persistent effect (~0.5s spread between best 
     and worst crews).
  2. SC has its own independent effect (stops under SC are slower by ~0.18s 
     regardless of team).
  3. Circuit has essentially no effect on stationary time.

METHODOLOGICAL LIMITATION -- SAMPLE SIZE:
stop_duration only has meaningful coverage from 2024 onward, and even then not 
for every stop -- final n=808 is much smaller than earlier questions' sample 
sizes. This limits what can be said about smaller effects, and 2024 (n=122) has 
notably less coverage than 2025 (n=680), so the analysis is really "2025-
dominant with 2024/2026 supporting data" rather than an evenly-weighted multi-
year study.

Pit stop duration differs significantly by team, and this difference survives when SC status and circuit are held constant, meaning team crew skill is a real, isolable effect, not an artifact of some teams pitting under different conditions than others. Separately, SC status has its own independent effect (stops under SC take ~0.18s longer regardless of team), and circuit differences don't meaningfully affect stationary time.

- [X] Does a slow pit stop reliably cost track position, or does pack density/traffic matter more? (correlate stop_duration against the position-swing data)

In [20]:

lap_dates_query = """
SELECT session_key, driver_number, lap_number, date_start
FROM silver_laps
WHERE date_start IS NOT NULL
"""
df_lap_dates = pd.read_sql(lap_dates_query, conn)

df_pits_pos = df_pits_clean.merge(
    df_lap_dates.rename(columns={'date_start':'pit_lap_date'}),
    on=['session_key','driver_number','lap_number'],
    how='left'
)

# Merge the NEXT lap's date_start too. Trick: add 1 to lap_number when merging on the lap_dates side.
df_lap_dates_next = df_lap_dates.copy()
df_lap_dates_next['lap_number'] = df_lap_dates_next['lap_number'] - 1  # so lap N in df_lap_dates matches lap N-1 here
df_pits_pos = df_pits_pos.merge(
    df_lap_dates_next.rename(columns={'date_start':'next_lap_date'}),
    on=['session_key','driver_number','lap_number'],
    how='left'
)

df_pits_pos.shape
df_pits_pos[['pit_lap_date','next_lap_date']].isna().sum()

pit_lap_date     41
next_lap_date    39
dtype: int64

In [21]:
df_pits_pos = df_pits_pos.dropna(subset=['pit_lap_date','next_lap_date']).copy()
df_pits_pos.shape

(767, 10)

In [22]:
pos_query = """
SELECT session_key, driver_number, date, position
FROM silver_position
"""
df_position = pd.read_sql(pos_query, conn)

df_position['date'] = pd.to_datetime(df_position['date'], format='ISO8601')
df_pits_pos['pit_lap_date'] = pd.to_datetime(df_pits_pos['pit_lap_date'], format='ISO8601')
df_pits_pos['next_lap_date'] = pd.to_datetime(df_pits_pos['next_lap_date'], format='ISO8601')


df_position = df_position.sort_values('date')

# Position BEFORE the pit (at start of pit lap)
df_pits_pos = df_pits_pos.sort_values('pit_lap_date')
df_pits_pos = pd.merge_asof(
    df_pits_pos,
    df_position.rename(columns={'position':'position_before'}),
    left_on='pit_lap_date', right_on='date',
    by=['session_key','driver_number'],
    direction='nearest'
).drop(columns='date')

# Position AFTER the pit (at start of next lap)
df_pits_pos = df_pits_pos.sort_values('next_lap_date')
df_pits_pos = pd.merge_asof(
    df_pits_pos,
    df_position.rename(columns={'position':'position_after'}),
    left_on='next_lap_date', right_on='date',
    by=['session_key','driver_number'],
    direction='nearest'
).drop(columns='date')

df_pits_pos['position_change'] = df_pits_pos['position_after'] - df_pits_pos['position_before']

df_pits_pos[['position_before','position_after','position_change']].describe()

,position_before,position_after,position_change
count,731.000000,731.000000,731.000000
mean,8.649795,9.660739,1.010944
std,5.168973,5.386165,1.842390
min,1.000000,1.000000,-8.000000
25%,4.000000,5.000000,0.000000
50%,8.000000,9.000000,1.000000
75%,13.000000,14.000000,1.000000
max,20.000000,20.000000,12.000000


In [25]:
intervals_query = """
SELECT session_key, driver_number, date, interval_seconds
FROM silver_intervals
WHERE interval_seconds IS NOT NULL
"""
df_intervals = pd.read_sql(intervals_query, conn)
df_intervals['date'] = pd.to_datetime(df_intervals['date'], format='ISO8601')
df_intervals = df_intervals.sort_values('date')

df_pits_pos = df_pits_pos.sort_values('pit_lap_date')
df_pits_pos = pd.merge_asof(
    df_pits_pos,
    df_intervals.rename(columns={'interval_seconds':'gap_ahead_at_pit'}),
    left_on='pit_lap_date', right_on='date',
    by=['session_key','driver_number'],
    direction='nearest'
).drop(columns='date')

df_pits_pos['gap_ahead_at_pit'].describe()
#df_pits_pos[['stop_duration','position_change','gap_ahead_at_pit']].isna().sum()

count    701.000000
mean       5.042616
std       28.857611
min        0.000000
25%        0.798000
50%        2.081000
75%        4.955000
max      755.121000
Name: gap_ahead_at_pit, dtype: float64

In [26]:
df_pits_pos = df_pits_pos.dropna(subset=['position_change','gap_ahead_at_pit']).copy()
df_pits_pos = df_pits_pos[df_pits_pos['gap_ahead_at_pit'] < 60].copy()  # tunable cutoff
df_pits_pos.shape
df_pits_pos[['stop_duration','position_change','gap_ahead_at_pit']].describe()

,stop_duration,position_change,gap_ahead_at_pit
count,664.000000,664.000000,664.000000
mean,2.756627,0.996988,4.017386
std,0.546662,1.833982,5.349279
min,1.800000,-8.000000,0.000000
25%,2.400000,0.000000,0.795500
50%,2.600000,1.000000,2.119000
75%,3.000000,1.000000,4.985250
max,4.600000,12.000000,41.014000


In [27]:
import statsmodels.formula.api as smf

model = smf.ols(
    'position_change ~ stop_duration + gap_ahead_at_pit',
    data=df_pits_pos
).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:        position_change   R-squared:                       0.025
Model:                            OLS   Adj. R-squared:                  0.022
Method:                 Least Squares   F-statistic:                     8.492
Date:                Sun, 19 Jul 2026   Prob (F-statistic):           0.000228
Time:                        11:56:01   Log-Likelihood:                -1336.0
No. Observations:                 664   AIC:                             2678.
Df Residuals:                     661   BIC:                             2691.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept            0.6234      0.365  

In [32]:
model_ext2 = smf.ols(
    'position_change ~ stop_duration + gap_ahead_at_pit + C(sc_active) + position_before',
    data=df_pits_pos
).fit()
print(model_ext2.summary())

                            OLS Regression Results                            
Dep. Variable:        position_change   R-squared:                       0.040
Model:                            OLS   Adj. R-squared:                  0.034
Method:                 Least Squares   F-statistic:                     6.920
Date:                Sun, 19 Jul 2026   Prob (F-statistic):           1.85e-05
Time:                        12:16:43   Log-Likelihood:                -1330.7
No. Observations:                 664   AIC:                             2671.
Df Residuals:                     659   BIC:                             2694.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
Intercept             0.7231      0.36


BASE MODEL (stop_duration + gap_ahead_at_pit):
  R² = 0.025
  stop_duration:    coef=+0.209, p=0.105 -- NOT significant
  gap_ahead_at_pit: coef=-0.051, p<0.001 -- significant but small
  Interpretation: BOTH predictors together explain only 2.5% of variance.


EXTENDED MODEL (+ position_before):
  R² = 0.040
  Adding "where in the field the driver was when pitting" barely moved R².
  

DIRECT ANSWERS TO THE QUESTION:
Q: "Does a slow pit stop reliably cost track position?"
   Not reliably. Stop_duration trends toward costing position (positive 
   coefficient in every model version) but never crosses the significance 
   threshold. Each extra second of stop time is associated with ~0.2 places 
   lost on average, but this effect is small enough that noise dominates it.

Q: "Or does pack density/traffic matter more?"
   Traffic (gap_ahead_at_pit) is the more statistically robust predictor of 
   the two -- significant across every model version, direction sensible 
   (larger gap = fewer places lost, coef=-0.05). But its practical effect size 
   is also small: ~0.05 places gained per second of extra gap.

DEEPER FINDING -- WHAT ACTUALLY DOMINATES:
Neither slow stops nor traffic pressure dominates position_change. Extending 
the model with SC status, lap number, and field position all together brought 
R² only from 0.025 to 0.040 -- meaning ~96% of what determines position_change 
through a pit cycle is NOT captured by any measurable single-stop characteristic.

The likely dominant factor -- which the analysis could not directly test -- is 
COMPETING PIT STOPS ON ADJACENT LAPS. Whether the driver directly ahead of you 
also pits on the same lap (you rejoin behind them, minimal change) or stays 
out for many more laps (you rejoin far behind them, multiple positions lost) 
is arguably the biggest single factor in pit-cycle position change. This is 
the F1 undercut/overcut dynamic, and encoding it requires cross-driver 
awareness at pit-lap granularity that isn't a simple regression variable.

TAKEAWAY:
The question set up a binary between two variables (slow stops vs. traffic). 
Between those two, traffic wins for statistical robustness. But the more 
important honest finding is that BOTH effects are small and NEITHER dominates 
-- pit-cycle position change is dominated by strategic interactions between 
drivers that this analysis couldn't directly measure.


- [X] Are disaster stops (Tukey-fence outliers) random, or concentrated in specific teams or circuits? (chi-square)

In [33]:
pit_query = """
SELECT p.session_key, p.driver_number, p.lap_number,
       p.stop_duration,
       gs.year, m.circuit_short_name,
       d.team_name
FROM silver_pit p
JOIN silver_sessions gs ON p.session_key = gs.session_key AND gs.session_name = 'Race'
JOIN silver_meetings m ON gs.meeting_key = m.meeting_key
JOIN silver_drivers d ON p.session_key = d.session_key AND p.driver_number = d.driver_number
WHERE p.stop_duration IS NOT NULL
"""
df_pits_full = pd.read_sql(pit_query, conn)
df_pits_full = normalize_team_names(df_pits_full)
df_pits_full = df_pits_full[df_pits_full['team_name'] != 'Cadillac'].copy()

# Tag disaster stops using the Tukey cutoff derived earlier (4.65s)
df_pits_full['is_disaster'] = (df_pits_full['stop_duration'] > 4.65).astype(int)

print(df_pits_full.shape)
print(df_pits_full['is_disaster'].sum(), 'disaster stops')
print(df_pits_full['is_disaster'].value_counts(normalize=True))

(909, 8)
101 disaster stops
is_disaster
0    0.888889
1    0.111111
Name: proportion, dtype: float64


In [34]:
from scipy.stats import chi2_contingency

table_team = pd.crosstab(df_pits_full['team_name'], df_pits_full['is_disaster'])
print(table_team)

chi2, p, dof, expected = chi2_contingency(table_team)
low_cells = (expected < 5).sum()
print(f"\nchi2={chi2:.3f}, p={p:.6f}, dof={dof}")
print(f"Expected cells < 5: {low_cells}")

is_disaster       0   1
team_name              
Alpine           80   6
Aston Martin     78  13
Ferrari          90   5
Haas F1 Team     80  15
McLaren          83   9
Mercedes         90   3
RB Family        80   8
Red Bull Racing  80  14
Sauber Family    68  12
Williams         79  16

chi2=19.940, p=0.018283, dof=9
Expected cells < 5: 0


In [35]:
table_circuit = pd.crosstab(df_pits_full['circuit_short_name'], df_pits_full['is_disaster'])
print(table_circuit)

chi2, p, dof, expected = chi2_contingency(table_circuit)
low_cells = (expected < 5).sum()
print(f"\nchi2={chi2:.3f}, p={p:.6f}, dof={dof}")
print(f"Expected cells < 5: {low_cells}")

is_disaster          0   1
circuit_short_name        
Austin              39   5
Baku                19   1
Catalunya           85   8
Hungaroring         25   2
Imola               36   1
Interlagos          37  11
Jeddah              16   3
Las Vegas           36   2
Lusail              33   8
Melbourne           40  11
Mexico City         38   8
Miami               34   1
Monte Carlo         37   2
Montreal            31   2
Monza               16   3
Sakhir              36   4
Shanghai            35   5
Silverstone         26   2
Singapore           18   3
Spa-Francorchamps   24   2
Spielberg           30   2
Suzuka              37   2
Yas Marina Circuit  45   9
Zandvoort           35   4

chi2=33.393, p=0.074464, dof=23
Expected cells < 5: 19


Why is RedBull Racing surprisingly high?  what else predicts disasters?

In [40]:
sc_query = """
SELECT DISTINCT session_key, lap_number
FROM silver_race_control
WHERE category = 'SafetyCar' AND lap_number IS NOT NULL
"""
df_sc = pd.read_sql(sc_query, conn)
df_sc['sc_active'] = 1

df_pits_full = df_pits_full.merge(df_sc, on=['session_key','lap_number'], how='left')
df_pits_full['sc_active'] = df_pits_full['sc_active'].fillna(0).astype(int)

In [41]:
# chi-square: sc_active vs is_disaster
table_sc = pd.crosstab(df_pits_full['sc_active'], df_pits_full['is_disaster'])
print(table_sc)

from scipy.stats import chi2_contingency
chi2, p, dof, expected = chi2_contingency(table_sc)
print(f"\nchi2={chi2:.3f}, p={p:.6f}")
print(f"Expected cells < 5: {(expected < 5).sum()}")

# also compute the disaster rate under each condition
print(f"\nDisaster rate under green flag: {df_pits_full[df_pits_full['sc_active']==0]['is_disaster'].mean():.1%}")
print(f"Disaster rate under SC:         {df_pits_full[df_pits_full['sc_active']==1]['is_disaster'].mean():.1%}")

is_disaster    0   1
sc_active           
0            696  89
1            112  12

chi2=0.154, p=0.694389
Expected cells < 5: 0

Disaster rate under green flag: 11.3%
Disaster rate under SC:         9.7%


In [42]:
df_pits_full = df_pits_full.sort_values(['session_key','driver_number','lap_number'])
df_pits_full['stop_number'] = df_pits_full.groupby(['session_key','driver_number']).cumcount() + 1

print(df_pits_full['stop_number'].value_counts().sort_index())

table_stop = pd.crosstab(df_pits_full['stop_number'], df_pits_full['is_disaster'])
print(table_stop)

chi2, p, dof, expected = chi2_contingency(table_stop)
print(f"\nchi2={chi2:.3f}, p={p:.6f}")
print(f"Expected cells < 5: {(expected < 5).sum()}")
print("\nDisaster rate by stop number:")
print(df_pits_full.groupby('stop_number')['is_disaster'].agg(['mean','count']))

stop_number
1    590
2    269
3     46
4      4
Name: count, dtype: int64
is_disaster    0   1
stop_number         
1            529  61
2            241  28
3             34  12
4              4   0

chi2=11.436, p=0.009587
Expected cells < 5: 2

Disaster rate by stop number:
                 mean  count
stop_number                 
1            0.103390    590
2            0.104089    269
3            0.260870     46
4            0.000000      4


In [43]:
df_pits_full['stop_number_capped'] = df_pits_full['stop_number'].clip(upper=3)

table_stop = pd.crosstab(df_pits_full['stop_number_capped'], df_pits_full['is_disaster'])
print(table_stop)

chi2, p, dof, expected = chi2_contingency(table_stop)
print(f"\nchi2={chi2:.3f}, p={p:.6f}")
print(f"Expected cells < 5: {(expected < 5).sum()}")

is_disaster           0   1
stop_number_capped         
1                   529  61
2                   241  28
3                    38  12

chi2=8.900, p=0.011676
Expected cells < 5: 0


METHOD:
Reused the 2024+ dataset from the crew-skill question. Tagged is_disaster = 
stop_duration > 4.65s (Tukey cutoff derived earlier). Ran chi-square tests of 
independence against multiple candidate factors: team, circuit, SC status, 
stop number within race. n=909 stops, 101 disasters (~11% base rate).

RESULTS BY FACTOR:

1. TEAM: chi2=19.94, p=0.018, expected cells<5: 0 -- SIGNIFICANT.
   Disaster rate varies from Mercedes (3.2%) to Williams (16.8%). The best 
   teams show 3-5% disaster rates; the worst cluster around 14-17%. Note the 
   worst-cluster teams (Aston Martin, Red Bull, Sauber, Haas, Williams) are 
   statistically indistinguishable from each other -- singling out any 
   individual team within this cluster isn't warranted.

2. CIRCUIT: chi2=33.39, p=0.074, expected cells<5: 19 (of 48). NOT reliable.
   Cannot conclude a circuit effect. Additionally, the earlier crew-skill 
   ANCOVA showed circuit had essentially no effect on median stop duration, 
   so a circuit-specific disaster effect would be inconsistent with that prior 
   finding.

3. SAFETY CAR STATUS: chi2=0.15, p=0.694, expected cells<5: 0 -- NOT 
   SIGNIFICANT. Disaster rate under SC (9.7%) is essentially identical to 
   under green flag (11.3%). Race chaos is NOT associated with disasters -- 
   contradicts the intuitive "chaos causes disasters" hypothesis.

4. STOP NUMBER WITHIN RACE: chi2=8.90, p=0.012, expected cells<5: 0 
   (after capping 4-stop rows into 3+ bucket) -- SIGNIFICANT.
   Stop 1: 10.3% disaster rate
   Stop 2: 10.4% disaster rate
   Stop 3+: 24.0% disaster rate (~2.4x higher than baseline)

INTERPRETATION -- WHAT MAKES A DISASTER:
Two independent, statistically real findings emerge:
  a) Team crew skill matters (some crews are 3-5x more reliable than others)
  b) 3rd+ pit stops within a race are ~2.4x more disaster-prone than 1st/2nd

Notably, race context (Safety Car) does NOT explain disasters. This narrows 
the mechanism significantly. Combined with the stop-number finding, the 
likely story is:
  - Most races are planned as 1-stop or 2-stop strategies
  - A 3rd+ stop typically means something has gone off-script -- damage, 
    unexpected tyre wear, forced compound switches, or unplanned strategic 
    responses
  - Off-script stops put crews under unusual conditions: unfamiliar timing, 
    unrehearsed tyre choices, higher time pressure, less muscle memory
  - This is where disasters happen, regardless of whether an SC is active

TAKEAWAY:
Disaster stops are NOT random. They are:
  1. Significantly concentrated in specific teams (crew skill effect)
  2. Significantly concentrated in 3rd+ stops within a race (off-script stops)
  3. NOT associated with Safety Car / chaos context
  4. Not reliably associated with specific circuits (insufficient sample)

The story that emerges: disasters happen when crews are executing UNUSUAL 
stops, not chaotic ones. Some crews (Mercedes, Ferrari) manage this variance 
much better than others.

METHODOLOGICAL NOTE:
The initial team ranking suggested Red Bull was "surprisingly" high given 
their fast median stop time. On closer look, Red Bull sits within a cluster 
of teams (14-17% disaster rate) that are statistically indistinguishable from 
each other; calling out Red Bull specifically wasn't warranted. Rewrote the 
conclusion to reflect this and reframed as a broader investigation of what 
predicts disasters generally -- which yielded the more informative 
stop-number finding above.